# Workshop 2B: ตรวจชิ้นงาน OK / NG
โจทย์: ในถาดต้องมีชิ้นสีแดงขนาดใหญ่ **ครบตามจำนวน** ถ้าขาดหรือเกิน = **NG**
ในโรงงาน เซนเซอร์จะสั่งถ่ายภาพเมื่อชิ้นงานมาถึง (Trigger)  ใน Notebook นี้ **รัน cell ข้อ 3 หนึ่งครั้ง = Trigger หนึ่งครั้ง**

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))   # ให้หาไฟล์ vision_camera.py เจอ

import cv2
import numpy as np
from vision_camera import show, grab_one, img_path, put_label
print("OpenCV", cv2.__version__, "พร้อมใช้งาน")

OpenCV 4.11.0 พร้อมใช้งาน


## 1. ตั้งค่างานตรวจ

In [ ]:
SOURCE   = "file"     # "file" | "emu" | "basler"
IMAGE    = "shapes.png"
EXPECT   = 5          # จำนวนชิ้นที่ต้องมี
H_LOW, H_HIGH = 0, 10
MIN_AREA = 1000
LOG_FILE = "inspection_log.csv"

## 2. ฟังก์ชันนับและตัดสิน

In [ ]:
def count_big(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([H_LOW, 50, 50]), np.array([H_HIGH, 255, 255]))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    boxes = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > MIN_AREA]
    return boxes

def judge(n):
    return "OK" if n == EXPECT else "NG"

## 3. Trigger: รัน cell นี้ซ้ำได้เรื่อยๆ
ลองเปลี่ยน `EXPECT` ใน cell ข้อ 1 เป็น 6 แล้วรันใหม่ จะได้ NG

In [ ]:
import csv, time
from datetime import datetime

t0 = time.perf_counter()
frame = grab_one(SOURCE, image=IMAGE)
boxes = count_big(frame)
verdict = judge(len(boxes))
ms = (time.perf_counter() - t0) * 1000

view = frame.copy()
for x, y, w, h in boxes:
    cv2.rectangle(view, (x, y), (x + w, y + h), (200, 0, 100), 3)
color = (0, 160, 0) if verdict == "OK" else (0, 0, 210)
cv2.rectangle(view, (0, 0), (view.shape[1], 50), color, -1)
cv2.putText(view, f"{verdict}  {len(boxes)}/{EXPECT}", (10, 38),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
new = not os.path.exists(LOG_FILE)
with open(LOG_FILE, "a", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    if new:
        w.writerow(["time", "verdict", "count", "expect", "ms"])
    w.writerow([now, verdict, len(boxes), EXPECT, f"{ms:.0f}"])

if verdict == "NG":                                   # เก็บภาพชิ้นงานเสียไว้ตรวจย้อนหลัง
    os.makedirs("ng_images", exist_ok=True)
    cv2.imwrite(f"ng_images/NG_{now.replace(':', '-')}.png", view)

print(f"{now}  ผล: {verdict}  พบ {len(boxes)} ชิ้น (ต้องมี {EXPECT})  ใช้เวลา {ms:.0f} ms")
show(view, titles=[verdict])

## 4. ดูบันทึกผลการตรวจ

In [ ]:
with open(LOG_FILE, encoding="utf-8-sig") as f:
    rows = list(csv.reader(f))[1:]
ok = sum(1 for r in rows if r[1] == "OK")
print(f"ตรวจทั้งหมด {len(rows)} ครั้ง   OK {ok}   NG {len(rows) - ok}")
for r in rows[-5:]:
    print(r)

## ลองเอง (ระดับ 2)
ให้เป็น NG ด้วย ถ้ามี **ชิ้นแดงเล็ก** ปนมา (ชิ้นที่พื้นที่น้อยกว่า `MIN_AREA`)
คำใบ้: นับ contour ทั้งหมด แล้วลบจำนวนชิ้นใหญ่

---
# ลงมือทำ: ยกระดับสถานีตรวจ OK / NG (4 ภารกิจ)
ข้างบนตรวจได้แค่ “นับครบ 5 ไหม”  สถานีจริงต้องบอกได้ว่า**เสียเพราะอะไร** และสรุปผลทั้งล็อตได้
แต่ละภารกิจมี `TODO` ให้แก้ และตัวตรวจขึ้น ✅ / ❌ ทันที  เฉลยอยู่ท้าย notebook

In [ ]:
def check(title, got, expected):
    ok = got == expected
    print(('✅ ผ่าน  ' if ok else '❌ ยัง   ') + title + f'   ได้ {got}  ต้องการ {expected}')
    return ok

EXPECT = 5                     # คืนค่าเดิม เผื่อเปลี่ยนไว้ตอนลองเอง
TRAYS = sorted(os.path.join(img_path('line'), f) for f in os.listdir(img_path('line')) if f.endswith('.png'))
print('ถาดทั้งหมด', len(TRAYS), 'ใบ:', [os.path.basename(p) for p in TRAYS])

## ภารกิจ 1: จับเศษชิ้นแตก
ชิ้นแดงที่**เล็กกว่า** `MIN_AREA` แต่ใหญ่กว่า 50 พิกเซล คือเศษของชิ้นที่แตก ต้องนับได้

In [ ]:
def count_chips(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([H_LOW, 50, 50]), np.array([H_HIGH, 255, 255]))
    cs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return 0                                       # TODO: นับ contour ที่พื้นที่อยู่ระหว่าง 50 ถึง MIN_AREA

check('ภารกิจ 1 เศษใน tray_03', count_chips(cv2.imread(img_path('line/tray_03.png'))), 2)
check('ภารกิจ 1 เศษใน tray_01', count_chips(cv2.imread(img_path('line/tray_01.png'))), 0)

## ภารกิจ 2: จับสิ่งแปลกปลอมสีส้ม
สีส้มอยู่ราว H 12–22  นับชิ้นสีส้มที่ใหญ่กว่า `MIN_AREA`

In [ ]:
def count_orange(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return 0                                       # TODO: สร้าง mask สีส้ม แล้วนับ contour ที่ใหญ่กว่า MIN_AREA

check('ภารกิจ 2 สีส้มใน tray_05', count_orange(cv2.imread(img_path('line/tray_05.png'))), 1)
check('ภารกิจ 2 สีส้มใน tray_01', count_orange(cv2.imread(img_path('line/tray_01.png'))), 0)

## ภารกิจ 3: บอกสาเหตุของ NG
คืนรหัสสาเหตุ ตามลำดับความสำคัญ: `MISSING` (ขาด) → `EXTRA` (เกิน) → `CHIP` (มีเศษ) → `FOREIGN` (มีสิ่งแปลกปลอม)
พนักงานหน้าไลน์ใช้รหัสนี้แก้ปัญหาได้ทันที ไม่ต้องเดาว่าเสียเพราะอะไร

In [ ]:
def inspect_full(img):
    n_big = len(count_big(img))
    n_chip = count_chips(img)
    n_orange = count_orange(img)
    if n_big < EXPECT:
        return 'NG', 'MISSING'
    # TODO: เพิ่มเงื่อนไข EXTRA, CHIP, FOREIGN ตามลำดับ
    return 'OK', 'OK'

reasons = [inspect_full(cv2.imread(p))[1] for p in TRAYS]
check('ภารกิจ 3 สาเหตุของถาด 1-6', reasons, ['OK', 'MISSING', 'CHIP', 'EXTRA', 'FOREIGN', 'OK'])

## ภารกิจ 4: สรุปผลทั้งล็อต
ตรวจถาดทั้ง 6 ใบ  คำนวณ **Yield** (% ที่ผ่าน) และเก็บภาพ NG แยกตามสาเหตุ ไว้ให้ QC ตรวจซ้ำ

In [ ]:
import shutil
shutil.rmtree('ng_by_reason', ignore_errors=True)
os.makedirs('ng_by_reason')
summary = {}
for p in TRAYS:
    verdict, reason = inspect_full(cv2.imread(p))
    summary[reason] = summary.get(reason, 0) + 1
    # TODO: ถ้า NG ให้บันทึกภาพด้วย cv2.imwrite ไปที่ ng_by_reason/<สาเหตุ>_<ชื่อไฟล์ถาด>

yield_pct = 0                                      # TODO: % ของถาดที่ OK จากทั้งหมด
print('สรุปตามสาเหตุ:', summary)
check('ภารกิจ 4 Yield (%)', round(yield_pct), 33)
check('ภารกิจ 4 ภาพ NG ที่เก็บไว้', len(os.listdir('ng_by_reason')), 4)

<details>
<summary><b>เฉลย (คลิกเพื่อเปิด)</b></summary>

**ภารกิจ 1:** `return sum(50 < cv2.contourArea(c) <= MIN_AREA for c in cs)`

**ภารกิจ 2:**
```python
mask = cv2.inRange(hsv, np.array([12, 50, 50]), np.array([22, 255, 255]))
cs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
return sum(cv2.contourArea(c) > MIN_AREA for c in cs)
```

**ภารกิจ 3:**
```python
if n_big > EXPECT:
    return 'NG', 'EXTRA'
if n_chip > 0:
    return 'NG', 'CHIP'
if n_orange > 0:
    return 'NG', 'FOREIGN'
```

**ภารกิจ 4:**
```python
if verdict == 'NG':
    cv2.imwrite(os.path.join('ng_by_reason', reason + '_' + os.path.basename(p)), cv2.imread(p))
...
yield_pct = summary.get('OK', 0) / len(TRAYS) * 100
```
</details>